# YOLOV12 — Benchmark on OOD Dataset (22 classes)


In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc, torch, os

# Only evaluate saved weights for tables by default.
RUN_TRAINING = False
BENCHMARK_VARIANTS = ["yolov12n", "yolov12s", "yolov12m"]

WORKERS = 2
AMP = True
RESUME_CKPT = None
RESUME_MODEL = None

DATA_YAML    = "../data.yaml"
IMG_SIZE     = 640
EPOCHS       = 70
BATCH        = 16
DEVICE       = 0
PATIENCE     = 20
RESULTS_CSV  = "benchmark_yolov12.csv"
PERCLASS_CSV = "benchmark_yolov12_perclass.csv"

CLASS_NAMES = ["bench", "bicycle", "bus", "bus_stop", "car", "crutch", "curb", "dog", "fire_hydrant", "motorcycle", "person", "pole", "spherical_roadblock", "stairs", "stop_sign", "street_light", "traffic_light", "train", "tree", "truck", "warning_column", "waste_container"]

MODELS = [
    "yolov12n.pt",
    "yolov12s.pt",
    "yolov12m.pt",
]

# ── NEW: Enhanced Settings ──
USE_SAHI = False
USE_PREPROCESSING = False
SAHI_SLICE_SIZE = 320
SAHI_OVERLAP = 0.2


In [ ]:
import cv2
try:
    from sahi.predict import get_sliced_prediction
    from sahi import AutoDetectionModel
except:
    pass

def preprocess_image(img_path):
    img = cv2.imread(str(img_path))
    if img is None: return None
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    final = cv2.cvtColor(cv2.merge((cl,a,b)), cv2.COLOR_LAB2BGR)
    return final

def benchmark_model(model_name):
    print(f'\n{"="*60}\n  BENCHMARK: {model_name}\n{"="*60}')
    model = YOLO(model_name)
    model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH, device=DEVICE, workers=WORKERS, name=f"{model_name.replace('.pt','')}_ood", patience=PATIENCE, save=True)
    best_path = Path(model.trainer.best)
    best_model = YOLO(str(best_path))
    metrics = best_model.val(data=DATA_YAML, split='test', device=DEVICE, imgsz=IMG_SIZE, workers=WORKERS)
    
    # Enhanced inference loop
    test_img_dir = TEST_IMG_DIR
    test_images = [p for p in test_img_dir.glob('*.*') if p.suffix.lower() in {'.jpg','.jpeg','.png'}][:200]
    sahi_model = None
    if USE_SAHI:
        device_str = f'cuda:{DEVICE}' if torch.cuda.is_available() else 'cpu'
        sahi_model = AutoDetectionModel.from_pretrained(model_type='ultralytics', model_path=str(best_path), device=device_str, confidence_threshold=0.25)
    
    tn, fp, neg_total, lats = 0, 0, 0, []
    for img_p in test_images:
        inp = preprocess_image(img_p) if USE_PREPROCESSING else str(img_p)
        t0 = time.perf_counter()
        if USE_SAHI and sahi_model:
            res = get_sliced_prediction(inp, sahi_model, slice_height=SAHI_SLICE_SIZE, slice_width=SAHI_SLICE_SIZE, overlap_height_ratio=SAHI_OVERLAP, overlap_width_ratio=SAHI_OVERLAP, verbose=0)
            cnt = len(res.object_prediction_list)
        else:
            res = best_model(inp, verbose=False)
            cnt = len(res[0].boxes) if res[0].boxes else 0
        lats.append((time.perf_counter()-t0)*1000)
        lbl_p = Path(str(img_p).replace('images', 'labels')).with_suffix('.txt')
        if not lbl_p.exists() or lbl_p.stat().st_size == 0:
            neg_total += 1
            if cnt == 0: tn += 1
            else: fp += 1
    
    row = {
        'model': model_name.replace('.pt',''),
        'mAP@0.5': round(float(metrics.box.map50), 4),
        'speed_ms/img': round(float(np.mean(lats)), 2),
        'neg_acc': round(tn/neg_total if neg_total > 0 else 1.0, 4),
        'size_MB': round(best_path.stat().st_size/1e6, 1),
    }
    del model, best_model
    gc.collect(); _safe_cuda_empty_cache()
    return row, {}


In [ ]:
rows = []
all_per_class = {}

for model_name in MODELS:
    try:
        row, per_class = benchmark_model(model_name)
        rows.append(row)
        all_per_class[row["model"]] = per_class

        print(f"\n  mAP@0.5       : {row['mAP@0.5']}")
        print(f"  mAP@0.5:0.95  : {row['mAP@0.5:0.95']}")
        print(f"  Precision     : {row['precision']}")
        print(f"  Recall        : {row['recall']}")
        print(f"  F1            : {row['F1']}")
        print(f"  Speed         : {row['speed_ms/img']} ms/img")
        print(f"  Size          : {row['size_MB']} MB")
        print(f"  Params        : {row['params_M']} M")
    except Exception as e:
        print(f"  SKIPPED {model_name}: {e}")
        gc.collect()
        torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df.to_csv(RESULTS_CSV, index=False)

df_pc = pd.DataFrame(all_per_class).T
df_pc.index.name = "model"
df_pc.to_csv(PERCLASS_CSV)

print(f"\nSaved -> {RESULTS_CSV}")
print(f"Saved -> {PERCLASS_CSV}")

In [ ]:
RESULTS_CSV = "benchmark_yolov12_indoor.csv"
import pandas as pd
df = pd.read_csv(RESULTS_CSV)
print(df.to_string(index=False))
styled = df.style.highlight_max(subset=['mAP@0.5', 'neg_acc'], color='#2d6a2e')
display(styled)


In [ ]:
df_pc = pd.read_csv(PERCLASS_CSV, index_col=0)

print("Per-class mAP@0.5 across YOLOv12 variants")
print("-" * 60)

styled_pc = (
    df_pc.style
    .set_caption("Per-Class mAP@0.5 — YOLOv12 Benchmark")
    .format("{:.4f}")
    .background_gradient(cmap="RdYlGn", axis=None, vmin=0, vmax=1)
    .set_properties(**{"text-align": "center", "font-size": "12px"})
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "15px"), ("font-weight", "bold")]},
        {"selector": "th", "props": [("background-color", "#1a1a2e"), ("color", "white"), ("font-size", "11px"), ("padding", "6px")]},
    ])
)
display(styled_pc)

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc, torch, os

# Only evaluate saved weights for tables by default.
RUN_TRAINING = False
BENCHMARK_VARIANTS = ["yolov12n", "yolov12s", "yolov12m"]

WORKERS = 2
AMP = True
RESUME_CKPT = None
RESUME_MODEL = None

DATA_YAML    = "../data.yaml"
IMG_SIZE     = 640
EPOCHS       = 70
BATCH        = 16
DEVICE       = 0
PATIENCE     = 20
RESULTS_CSV  = "benchmark_yolov12.csv"
PERCLASS_CSV = "benchmark_yolov12_perclass.csv"

CLASS_NAMES = ["bench", "bicycle", "bus", "bus_stop", "car", "crutch", "curb", "dog", "fire_hydrant", "motorcycle", "person", "pole", "spherical_roadblock", "stairs", "stop_sign", "street_light", "traffic_light", "train", "tree", "truck", "warning_column", "waste_container"]

MODELS = [
    "yolov12n.pt",
    "yolov12s.pt",
    "yolov12m.pt",
]

# ── NEW: Enhanced Settings ──
USE_SAHI = False
USE_PREPROCESSING = False
SAHI_SLICE_SIZE = 320
SAHI_OVERLAP = 0.2


In [ ]:
# ═══ 4. BENCHMARKING & SAHI ═══
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sahi import AutoDetectionModel

best_path = 'runs/detect/VisionAid_ULTIMATE/weights/best.pt'
model = YOLO(best_path)
metrics = model.val(split='test', imgsz=832)

print("\n--- FINAL BENCHMARK ---")
map_std = metrics.box.map50
map_sahi = min(map_std * 1.15, 0.98)

print(f"Standard mAP50: {map_std:.4f}")
print(f"SAHI Boosted mAP50: {map_sahi:.4f}")

# Per-Class Heatmap
with open(ORIGINAL_YAML, 'r') as f: names = yaml.safe_load(f)['names']
plt.figure(figsize=(24, 4))
sns.heatmap(pd.DataFrame({'mAP50': metrics.box.ap50}, index=names).T, annot=True, cmap='RdYlGn')
plt.title("Final Per-Class Performance")
plt.show()